# Case 900 VDI transfer validation

This notebook reports only the predeclared F36-like/F52-like transfer test using the verified ASHRAE 140 Case 900 layers. It does not perform a factorial search. Layer-faithful outputs are stored separately from the historical scaled-cp proxy results.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
HERE=Path.cwd().resolve()
if HERE.name!='2_vdi': HERE=HERE/'2_validation/_BESTEST/2_vdi'
sys.path.insert(0,str(HERE/'doc'))
import bestest_vdi_transfer_engine as engine
RUN_SIMULATIONS=False
OUT=HERE/'results/case900_vdi_transfer_ashrae140_layers'

## Case 600 transfer-engine equivalence gate

The generalized runner must reproduce the stored Case 600 F36/F52 result before Case 900 is regenerated. This confirms that the Case 900-only material mapping does not alter Case 600 behavior.

In [2]:
defaults=engine._configure_defaults('900')
traceability=engine.construction_traceability(defaults)
display(traceability.round(9))
if RUN_SIMULATIONS:
    replay=engine.verify_case600_replay(tolerance=1e-9)
    summary,hourly,layers,traceability=engine.run_case900(OUT)
else:
    summary=pd.read_csv(OUT/'case900_vdi_transfer_summary.csv')
    hourly=pd.read_csv(OUT/'case900_vdi_transfer_hourly.csv')
    layers=pd.read_csv(OUT/'case900_layer_capacity_audit.csv')
    traceability=pd.read_csv(OUT/'case900_construction_traceability.csv')
assert len(summary)==2 and len(hourly)==2*8760
assert np.isfinite(summary.select_dtypes('number')).all().all()
assert summary.maximum_balance_residual_W.max()<=1e-9

,construction_convention,wall_layers,floor_layers,roof_layers,raw_layer_capacity_J_K,VDI_reduced_AW_resistance_K_W,VDI_reduced_AW_capacitance_J_K
0,ashrae140_layers,wood siding + foam insulation + concrete block,massless insulation [ASHRAE rho=0 cp=0; positi...,roof deck + fiberglass quilt + plasterboard,1.547995e+07,0.000385,1.449967e+07


## Construction and input traceability

The pre-simulation table above records the primary construction convention, wall/floor/roof chains, raw layer capacity, and VDI-reduced AW resistance/capacitance. Prescribed component U-values remain separate steady-state transmission inputs.

In [3]:
audit=pd.read_csv(OUT/'case900_input_translation_audit.csv')
checklist=pd.read_csv(OUT/'case900_eight_point_checklist.csv')
assert np.isclose(layers.total_C_J_K.sum(),15_479_951.712,atol=1e-6)
display(audit)
display(layers.round(6))
display(checklist)

,item,current_setting,classification,source_provenance
0,geometry,48 m2; 129.6 m3; same surfaces,unchanged from Case 600,inputs/case900.json
1,construction layers,ASHRAE 140 Case 900 wall/floor; Case 600 roof,benchmark-prescribed change,ASHRAE 140 Table 7-27; detailed Modelica records
2,prescribed U-values,wall .53; roof .33; floor .038 W/m2K,unchanged from Case 600,inputs/case900.json
3,thermal mass/capacitance,layer-derived by VDI dynamic reduction,benchmark-prescribed change,verified physical layer chains
4,effective mass area,"2.43 x 48 = 116.64 m2; recorded, not applied",unresolved provenance,no direct no-IW VDI mapping
5,glazing,12 m2 south; U 3.1; g .769,unchanged from Case 600,inputs/case900.json
6,infiltration/ventilation,0.414 1/h; no mechanical ventilation,unchanged from Case 600,repository comparator; canonical provenance open
7,internal gains,200 W; 50% air/50% AW,implementation assumption,comparator-consistent no-IW projection
8,solar properties,native VDI; absorptance .6; g .769,unchanged from Case 600,candidate implementation
9,setpoints/control,20/27 C; ideal air HVAC,unchanged from Case 600,benchmark/comparator


,assembly,area_m2,layer_R_m2K_W,layer_C_J_m2K,total_C_J_K
0,wall,63.6,1.797864,145154.000,9231794.400
1,roof,48.0,2.993214,18169.944,872157.312
2,floor,48.0,25.245796,112000.000,5376000.000


,case,category,verification_question,current_setting,source_provenance
0,900,IW topology,Is the intended topology present and are absen...,no-IW,Case600 candidate
1,900,Solar/source allocation,Which nodes receive solar and is allocation na...,F36/F52 variants,Case600 candidate
2,900,Envelope convention,Which inputs control transmission and dynamics...,ASHRAE 140 layers; prescribed U-values retained,ASHRAE 140 Table 7-27 + detailed Modelica records
3,900,Inside heat-transfer treatment,Which total convention is active and how is it...,total h_i=8 W/m2K,VDI-side convention
4,900,Exterior long-wave treatment,Is long-wave native or harmonised and what is ...,harmonised,Case600 candidate
5,900,Internal-gain allocation,Which nodes receive gains and what is the sour...,50% air / 50% AW,comparator projection
6,900,Airflow/infiltration provenance,"What airflow quantity and provenance are used,...",0.414 1/h,repository comparator; canonical source open
7,900,"Geometry, schedules and controls","Do geometry, schedules, setpoints, availabilit...",benchmark geometry/schedules/controls,canonical case record


## Annual heating/cooling and BESTEST checks

Distances are measured to the accepted intervals, not their centres.

In [4]:
cols=['configuration','heating_MWh','cooling_MWh','heating_in_range','cooling_in_range','both_in_range','D_MWh_norm','maximum_balance_residual_W']
display(summary[cols].round(6))

,configuration,heating_MWh,cooling_MWh,heating_in_range,cooling_in_range,both_in_range,D_MWh_norm,maximum_balance_residual_W
0,F36-like,2.272013,2.544784,True,True,True,0.0,0.0
1,F52-like,2.259676,2.596246,True,True,True,0.0,0.0


## Stop/go decision

The wider 6XX/9XX sequence remains stopped. This notebook records only the layer-faithful Case 900 F36-like/F52-like results and their checks against the existing bounds; mass-area and surface-coupling fidelity are reserved for Task 2.